# Guide to Extracting Data from the IMDb Movies by Year Dataset

This notebook does two things with pandas:

1. **It shows how to use the dataset on Kaggle:** reading the yearly files from `/kaggle/input` with the right type for every column, and parsing the JSON columns.
2. **It builds a new dataset:** one CSV with all the years combined, saved to `/kaggle/working`.

## The dataset in brief

- **107,000 feature films:** the top 1,000 per release year from 1920 to 2026, ranked by US box office.
- **One file per year:** `merged_movies_data_<year>.csv`. Every file has the same 98 columns in the same order, keyed by `imdb_id`.
- **List columns hold JSON text,** for example `["Drama", "Romance"]`.
- **Missing values are empty cells.** Words such as `none` (a parental-guide severity) are real values, not missing ones.

The dataset's `README.md` describes every column.

## Output

**`imdb_movies_1920_2026.csv`**: every film from every year, one row per film, sorted by year and then by box-office rank. It has the same 98 columns and the same value formats as the yearly files.

<h3 style="font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif; color: #2c3e50; margin-top: 40px;">Contact Me</h3>

<p style="font-size: 16px; color: #555;">
  If you notice anything lacking, spot an issue with this notebook, or have suggestions for improvements, feel free to reach out through any of the platforms below:
</p>

<table style="font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif; border-collapse: collapse; width: 100%; max-width: 600px;">
  <tbody>
    <tr>
      <td style="padding: 10px;">
        <a href="https://raedaddala.github.io/" target="_blank">
          <img src="https://img.shields.io/badge/Website-raedaddala.github.io-222222?style=for-the-badge&logo=githubpages" alt="Website Badge"/>
        </a>
      </td>
    </tr>
    <tr>
      <td style="padding: 10px;">
        <a href="https://www.linkedin.com/in/addalaraed/" target="_blank">
          <img src="https://img.shields.io/badge/LinkedIn-Raed_Addala-blue?style=for-the-badge&logo=linkedin" alt="LinkedIn Badge"/>
        </a>
      </td>
    </tr>
    <tr>
      <td style="padding: 10px;">
        <a href="https://www.researchgate.net/profile/Raed-Addala" target="_blank">
          <img src="https://img.shields.io/badge/ResearchGate-Raed_Addala-00CCBB?style=for-the-badge&logo=researchgate&logoColor=white" alt="ResearchGate Badge"/>
        </a>
      </td>
    </tr>
    <tr>
      <td style="padding: 10px;">
        <a href="https://x.com/AddalaRaed" target="_blank">
          <img src="https://img.shields.io/badge/Twitter-@AddalaRaed-1DA1F2?style=for-the-badge&logo=twitter" alt="Twitter Badge"/>
        </a>
      </td>
    </tr>
    <tr>
      <td style="padding: 10px;">
        <a href="mailto:addala.raed@gmail.com">
          <img src="https://img.shields.io/badge/Gmail-addala.raed@gmail.com-D14836?style=for-the-badge&logo=gmail" alt="Gmail Badge"/>
        </a>
      </td>
    </tr>
    <tr>
      <td style="padding: 10px;">
        <a href="https://github.com/RaedAddala" target="_blank">
          <img src="https://img.shields.io/badge/GitHub-RaedAddala-181717?style=for-the-badge&logo=github" alt="GitHub Badge"/>
        </a>
      </td>
    </tr>
  </tbody>
</table>

## 1. Setup

Attach the dataset first, with **Add Input** (`raedaddala/imdb-movies-by-year-1920-2026`). Kaggle mounts it read-only under `/kaggle/input`, and files written to `/kaggle/working` become the notebook's output.

In [1]:
import json
from pathlib import Path

import pandas as pd

pd.set_option("display.max_colwidth", 60)
print(f"pandas {pd.__version__}")

pandas 3.1.1


In [2]:
INPUT_DIR = Path("/kaggle/input/imdb-movies-by-year-1920-2026")
OUTPUT_DIR = Path("/kaggle/working")

## 2. Find the yearly files

In [3]:
files = sorted(INPUT_DIR.glob("merged_movies_data_*.csv"))
if not files:
    raise FileNotFoundError(f"No yearly files in {INPUT_DIR}. Attach the dataset "
                            "raedaddala/imdb-movies-by-year-1920-2026 with 'Add Input'.")
years = [int(path.stem.rsplit("_", 1)[1]) for path in files]
print(f"{len(files)} yearly files, {min(years)}-{max(years)}, "
      f"{sum(path.stat().st_size for path in files) / 1e6:,.0f} MB")

107 yearly files, 1920-2026, 642 MB


## 3. Load them with the right types

Letting `read_csv` guess the types goes wrong in two ways:

- **Whole numbers with gaps become floats.** A plain `int64` column cannot hold a missing value, so a column such as `metascore` comes back as floats (`74.0`). The nullable `Int64` type keeps them whole.
- **Some words are read as missing.** By default pandas treats `none`, a parental-guide severity, and several other words as missing values.

So `read_text` reads every column as text, with only empty cells counted as missing, and `apply_types` then gives each column its documented type:

| Columns | Type |
| --- | --- |
| Counts, years, ranks, money amounts | `Int64` |
| `imdb_rating`, `runtime_minutes`, `gross_to_budget_ratio` | `Float64` |
| `is_adult` | `boolean` |
| `first_release_date`, `opening_weekend_end_date` | `datetime64` |
| `scraped_at` | `datetime64` in UTC |
| `release_date` | Text, because it can be partial: `1995`, `1995-05` or `1995-05-17` |
| The JSON columns and everything else | Text |

The conversions are strict: a value that does not fit its type raises an error instead of quietly becoming a missing value.

In [4]:
MONEY_FIELDS = ["budget", "gross_us_canada", "gross_worldwide", "opening_weekend_us_canada"]
INT_COLUMNS = [
    "search_year", "listing_rank", "release_year", "runtime_seconds", "imdb_votes", "metascore",
    "top_rated_rank", "user_reviews_count", "critic_reviews_count", *[f"{field}_amount" for field in MONEY_FIELDS],
    "award_wins", "award_nominations", "prestigious_award_wins", "prestigious_award_nominations",
    "keywords_total", "filming_locations_total", "certificates_total", "cast_total", "release_dates_total",
    "akas_total", "release_decade", "release_month", "gross_minus_budget", "genre_count", "country_count",
    "language_count", "cast_count",
]
FLOAT_COLUMNS = ["imdb_rating", "runtime_minutes", "gross_to_budget_ratio"]
BOOL_COLUMNS = ["is_adult"]
DATE_COLUMNS = ["first_release_date", "opening_weekend_end_date"]
DATETIME_COLUMNS = ["scraped_at"]


def read_text(path):
    """A yearly (or combined) file with every column as text; only empty cells count as missing."""
    return pd.read_csv(path, dtype=str, keep_default_na=False, na_values=[""])


def apply_types(frame):
    """Convert the text columns to their documented types (JSON columns stay text)."""
    typed = frame.copy()
    for column in INT_COLUMNS:
        typed[column] = pd.to_numeric(typed[column], dtype_backend="numpy_nullable").astype("Int64")
    for column in FLOAT_COLUMNS:
        typed[column] = pd.to_numeric(typed[column], dtype_backend="numpy_nullable").astype("Float64")
    for column in BOOL_COLUMNS:
        unexpected = set(typed[column].dropna()) - {"True", "False"}
        if unexpected:
            raise ValueError(f"{column} holds values other than True/False: {sorted(unexpected)[:5]}")
        typed[column] = typed[column].map({"True": True, "False": False}).astype("boolean")
    for column in DATE_COLUMNS:
        typed[column] = pd.to_datetime(typed[column], format="%Y-%m-%d")
    for column in DATETIME_COLUMNS:
        typed[column] = pd.to_datetime(typed[column], format="%Y-%m-%dT%H:%M:%S%z", utc=True)
    return typed

In [5]:
movies_text = pd.concat([read_text(path) for path in files], ignore_index=True)
movies = apply_types(movies_text)
print(f"{len(movies):,} rows x {movies.shape[1]} columns")
display(movies.dtypes.astype(str).value_counts().rename_axis("type").rename("columns").to_frame())

107,000 rows x 98 columns


,columns
type,
str,61
Int64,30
Float64,3
datetime64[us],2
boolean,1
"datetime64[us, UTC]",1


## 4. Keep one row per film

IMDb's yearly search can return the same film in two years, and it happens once in this dataset. The row kept is the one whose search year matches the film's release year, or is closest to it. Ties go to the most recently scraped row. The table is then put back in file order: year, then box-office rank.

`movies_text` keeps the same rows as `movies`, still as the original text, for the CSV saved at the end.

In [6]:
duplicates = movies[movies["imdb_id"].duplicated(keep=False)]
display(duplicates[["imdb_id", "title", "search_year", "release_year", "scraped_at"]].reset_index(drop=True))

kept = (
    movies.assign(year_gap=(movies["search_year"] - movies["release_year"]).abs())
    .sort_values(["imdb_id", "year_gap", "scraped_at"], ascending=[True, True, False], na_position="last")
    .drop_duplicates("imdb_id")
    .sort_values(["search_year", "listing_rank"])
    .index
)
movies = movies.loc[kept]
movies_text = movies_text.loc[kept]
kept_from = movies.loc[movies["imdb_id"].isin(duplicates["imdb_id"]), "search_year"].tolist()
print(f"{duplicates['imdb_id'].nunique()} film(s) appeared in two yearly files; kept the row from {kept_from}. "
      f"{len(movies):,} films remain.")

,imdb_id,title,search_year,release_year,scraped_at
0,tt0061876,The Commissar,1967,1967,2026-10-14 05:26:37+00:00
1,tt0061876,The Commissar,1987,1987,2026-11-06 06:44:22+00:00


1 film(s) appeared in two yearly files; kept the row from [1987]. 106,999 films remain.


## 5. Parse the JSON columns

35 columns hold JSON text. `parse_json` turns a column into Python objects, and missing values stay missing:

| Columns | Each value becomes |
| --- | --- |
| `genres`, `interests`, `keywords`, `countries_of_origin`, `languages`, `production_companies`, `filming_locations`, `similar_movie_ids`, `similar_movie_titles`, `credits_incomplete`, and every credit list with its `_ids` list | A list of strings |
| `rating_histogram` | A list of 10 integers: the votes for ratings 1 to 10 |
| `cast_characters` | A list of lists: the characters each cast member played, parallel to `cast` |
| `certificates_by_country` | A list of `{country, rating, notes}` dicts |
| `release_dates` | A list of `{country, date, note}` dicts |
| `akas` | A list of `{country, title, note}` dicts |

The film with the most IMDb votes, with a few of its JSON columns parsed:

In [7]:
def parse_json(text):
    """JSON text to Python lists and dicts; missing stays missing. Always an object column, even when all missing."""
    return text.map(json.loads, na_action="ignore").astype(object)


example = movies.loc[[movies["imdb_votes"].idxmax()]]
print(f"{example['imdb_id'].iloc[0]}: {example['title'].iloc[0]} ({example['release_year'].iloc[0]})")
parsed = example[["genres", "rating_histogram", "directors", "cast", "cast_characters", "release_dates"]].apply(parse_json).iloc[0]
display(pd.DataFrame({"elements": parsed.map(len), "first elements": parsed.map(lambda values: values[:3])}))

tt0111161: The Shawshank Redemption (1994)


,elements,first elements
genres,1,[Drama]
rating_histogram,10,"[43694, 5698, 6183]"
directors,1,[Frank Darabont]
cast,74,"[Tim Robbins, Morgan Freeman, Bob Gunton]"
cast_characters,74,"[[Andy Dufresne], [Ellis Boyd 'Red' Redding], [Warden No..."
release_dates,72,"[{'country': 'Canada', 'date': '1994-09-10', 'note': 'To..."


## 6. Save all the years as one CSV

`movies_text` holds the kept films exactly as they appear in the yearly files, so the new file uses the same formats: `True`/`False`, ISO dates, and JSON text for the list columns. It goes to `/kaggle/working`, which makes it this notebook's output. To use it in another notebook, add this notebook's output as an input.

In [8]:
output = OUTPUT_DIR / f"imdb_movies_{min(years)}_{max(years)}.csv"
movies_text.to_csv(output, index=False, lineterminator="\n")
print(f"Saved {output.name}: {len(movies_text):,} films x {movies_text.shape[1]} columns, "
      f"{output.stat().st_size / 1e6:,.0f} MB")

Saved imdb_movies_1920_2026.csv: 106,999 films x 98 columns, 642 MB


The new file loads exactly like a yearly file: `apply_types(read_text(path))`. As a last check, it gives back the table built above.

In [9]:
reloaded = apply_types(read_text(output))
pd.testing.assert_frame_equal(reloaded, movies.reset_index(drop=True))
print(f"{output.name} loads back identical: {len(reloaded):,} films x {reloaded.shape[1]} columns.")

imdb_movies_1920_2026.csv loads back identical: 106,999 films x 98 columns.
